# ONNX Model Conversion for Cross-Platform Deployment
## AIAT 122 – Deep Learning

## Learning objectives
- Export a PyTorch model to ONNX format.
- Run inference with ONNX Runtime to verify the exported model.

**Where is this used in real life?** Mobile apps, edge devices, and web browsers often run models in different frameworks. **We use ONNX to deploy models across platforms** instead of locking to one framework because ONNX is a common interchange format; the same model can run on iOS (Core ML), Android, and in the browser.

**Prerequisites:** Python 3.8+, PyTorch basics. Run the pip cell first if needed.


## Short theory
- **ONNX** = Open Neural Network Exchange: a format that many frameworks can export to and run.
- **Export:** PyTorch `torch.onnx.export()` produces a `.onnx` file given a model and example input.
- **Inference:** ONNX Runtime loads the file and runs inference without PyTorch.
- **Why we use ONNX:** One trained model can run on mobile, edge, and web without rewriting.

## Inputs & Outputs
**Inputs:** PyTorch, a small model (built here), and ONNX Runtime.  
**Outputs:** An ONNX file, and inference result from ONNX Runtime. Run time: under ~2 min.


In [1]:
%pip install torch onnx onnxruntime -q
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
print("✅ Setup complete!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-macos 2.13.0 requires typing-extensions<4.6.0,>=3.6.6, but you have typing-extensions 4.13.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
✅ Setup complete!


## Part 1: Create PyTorch Model


In [2]:
# Simple PyTorch model for demo (we use ONNX to run it without PyTorch later)
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = SimpleModel()
model.eval()
print("✅ Model created.")
 


✅ Model created.


## Part 2: Convert to ONNX


In [3]:
# Create dummy input and export to ONNX
dummy_input = torch.randn(1, 10)
onnx_path = "simple_model.onnx"
torch.onnx.export(model, dummy_input, onnx_path, input_names=["input"], output_names=["output"])
print(f"✅ Exported to {onnx_path}")


✅ Exported to simple_model.onnx


## Part 3: Run Inference with ONNX Runtime


In [4]:
# Run inference with ONNX Runtime (no PyTorch needed)
session = ort.InferenceSession(onnx_path)
input_data = dummy_input.numpy()
input_name = session.get_inputs()[0].name
outputs = session.run(None, {input_name: input_data})
print("✅ ONNX inference successful!")
print("Output shape:", outputs[0].shape)
print("Sample output:", outputs[0][:2])
print("\nIn real life this .onnx file can run on: iOS (Core ML), Android, Web (ONNX.js), Edge (ONNX Runtime).")

✅ ONNX inference successful!
Output shape: (1, 1)
Sample output: [[0.04240617]]

In real life this .onnx file can run on: iOS (Core ML), Android, Web (ONNX.js), Edge (ONNX Runtime).


## 🧩 Mini-exercise | تمرين مصغر

**Try it:** Export the same model with a different opset version (e.g. opset_version=14) and run inference again. Check that the ONNX Runtime output still matches.

---

## Summary
**What you did**
- Built a small PyTorch model and exported it to ONNX.
- Ran inference with ONNX Runtime to verify the export.

**In real life you'd also:** Convert to platform-specific formats (e.g. Core ML for iOS), benchmark latency, and tune for edge devices.

**The main idea:** ONNX is a cross-platform format; export once, run on many runtimes.

**Next:** `04_model_pruning.ipynb` shows how to reduce model size by pruning weights.